# Dense Human Feedback Learning (DHFL) Segmentation Prototype

This version focuses on the strict adapter path you asked to check:

```python
F = backbone(x)
delta = adapter(F)
F_prime = F + delta
out = decoder(F_prime)
```

The adapter does **not** receive the human error map as an input in this version.
Human correction affects the adapter through gradient updates after a corrected
mask is available.

Feedback signal:

\[
E = G - \sigma(P)
\]

where \(P\) is the predicted logit map and \(G\) is the corrected mask. In this
notebook, \(E\) is used as an optional pixel-weighting signal during the
adapter-only update, not as an adapter input.


## Experiment Design

Main comparisons:

| Experiment | Adapter | Feedback update | Error-weighted correction loss |
|---|---:|---:|---:|
| A | no | no | no |
| B | yes | no | no |
| C | yes | yes | no |
| D | yes | yes | yes |

If feedback improves the correction/feedback set but not the held-out test set,
that suggests the adapter is learning from specific corrected samples but not
generalizing yet.


In [2]:
import copy
import math
from dataclasses import dataclass
from itertools import cycle
from typing import Optional

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch import Tensor
    from torch.utils.data import DataLoader, Dataset
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "PyTorch is required for this DHFL notebook. Install torch in this notebook kernel "
        "before running the model cells, for example: pip install torch"
    ) from exc


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

device: cpu


## Model Components

The forward path is intentionally simple:

```python
F = backbone(x)
F_prime, delta = adapter(F)
logits = decoder(F_prime)
```

Tensor contract:

```text
x:          [B, 1, H, W]
F:          [B, C, H/4, W/4]
delta:      [B, C, H/4, W/4]
F_prime:    [B, C, H/4, W/4]
logits:     [B, 1, H, W]
```

In [4]:
def _group_count(channels: int) -> int:
    for groups in (8, 4, 2, 1):
        if channels % groups == 0:
            return groups
    return 1


class ConvBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, stride: int = 1) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False),
            nn.GroupNorm(_group_count(out_channels), out_channels),
            nn.GELU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.GroupNorm(_group_count(out_channels), out_channels),
            nn.GELU(),
        )

    def forward(self, x: Tensor) -> Tensor:
        return self.net(x)


class SimpleBackbone(nn.Module):
    """Small stride-4 encoder used only for prototype testing."""

    downsample_factor = 4

    def __init__(self, in_channels: int = 1, base_channels: int = 24, feature_channels: int = 96) -> None:
        super().__init__()
        self.feature_channels = feature_channels
        self.net = nn.Sequential(
            ConvBlock(in_channels, base_channels),
            ConvBlock(base_channels, base_channels * 2, stride=2),
            ConvBlock(base_channels * 2, feature_channels, stride=2),
        )

    def forward(self, x: Tensor) -> Tensor:
        return self.net(x)


class IdentityAdapter(nn.Module):
    """No-adapter baseline: F' = F."""

    def forward(self, features: Tensor, error_map: Optional[Tensor] = None):
        return features, torch.zeros_like(features)


class ErrorAwareResidualAdapter(nn.Module):
    """Residual feature adapter: F_refined = F + gate * Delta.
 
    Inference (error_map=None):
        error_embed = zeros  (no signal available)
        gate = 1.0           ← adapter ALWAYS runs at inference;
                               its learned weights affect the output
        delta = delta_net(cat([norm(F), zeros]))
 
    Feedback training (error_map=E):
        error_embed = encoder(E)              ← tells adapter WHERE errors are
        gate = 1 - exp(-5 * |E|)             ← 0 when prediction already correct,
                                                grows toward 1 for large errors
        delta = gate * delta_net(cat([norm(F), error_embed]))
 
    The loss is additionally scaled by correction_signal (in feedback_step),
    giving a second path for gradient → 0 when E → 0.
    """
 
    def __init__(
        self,
        channels: int,
        error_channels: int = 1,
        error_embed_channels: int = 8,
        bottleneck_channels: int = 32,
        zero_init: bool = True,
    ) -> None:
        super().__init__()
        self.error_embed_channels = error_embed_channels
        self.feature_norm = nn.GroupNorm(_group_count(channels), channels)
        self.error_dropout = nn.Dropout(p=0.3)
 
        self.error_encoder = nn.Sequential(
            nn.Conv2d(error_channels, error_embed_channels, kernel_size=3, padding=1),
            nn.GELU(),
            nn.Conv2d(error_embed_channels, error_embed_channels, kernel_size=3, padding=1),
            nn.GELU(),
        )
 
        self.delta_net = nn.Sequential(
            nn.Conv2d(channels + error_embed_channels, bottleneck_channels, kernel_size=1),
            nn.GELU(),
            nn.Conv2d(bottleneck_channels, bottleneck_channels, kernel_size=3, padding=1),
            nn.GELU(),
            nn.Conv2d(bottleneck_channels, channels, kernel_size=1),
        )
 
        if zero_init:
            last = self.delta_net[-1]
            nn.init.zeros_(last.weight)
            nn.init.zeros_(last.bias)
 
    def _encode_error(self, features: Tensor, error_map: Optional[Tensor]) -> Tensor:
        """Return error embedding at feature spatial resolution."""
        batch_size, _, height, width = features.shape
 
        if error_map is None:
            return features.new_zeros(batch_size, self.error_embed_channels, height, width)
 
        if error_map.ndim != 4:
            raise ValueError(f"error_map must be [B, C, H, W], got {tuple(error_map.shape)}")
        if error_map.shape[0] != batch_size:
            raise ValueError("error_map batch size must match features")
 
        resized = F.interpolate(error_map, size=(height, width), mode="bilinear", align_corners=False)
        return self.error_dropout(self.error_encoder(resized))
 
    def forward(self, features: Tensor, error_map: Optional[Tensor] = None):
        error_embed = self._encode_error(features, error_map)
 
        adapter_input = torch.cat(
            [self.feature_norm(features), error_embed],
            dim=1,
        )
        delta = self.delta_net(adapter_input)
 
        if error_map is None:
            gate = 1.0
        else:
            error_strength = error_map.abs().mean(dim=(1, 2, 3), keepdim=True)
            gate = 1.0 - torch.exp(-5.0 * error_strength)
 
        delta = gate * delta
        refined = features + delta
        return refined, delta

class UNetStyleDecoder(nn.Module):
    """Lightweight decoder that upsamples stride-4 features to mask logits."""

    def __init__(self, in_channels: int, out_channels: int = 1, hidden_channels: int = 64) -> None:
        super().__init__()
        mid_channels = max(hidden_channels // 2, 16)
        low_channels = max(hidden_channels // 4, 8)
        self.net = nn.Sequential(
            ConvBlock(in_channels, hidden_channels),
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),
            ConvBlock(hidden_channels, mid_channels),
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),
            ConvBlock(mid_channels, low_channels),
            nn.Conv2d(low_channels, out_channels, kernel_size=1),
        )

    def forward(self, features: Tensor) -> Tensor:
        return self.net(features)


@dataclass(frozen=True)
class ForwardDebug:
    logits: Tensor
    features: Tensor
    refined_features: Tensor
    delta_features: Tensor


class DHFLSegmenter(nn.Module):
    def __init__(
        self,
        in_channels: int = 1,
        out_channels: int = 1,
        base_channels: int = 24,
        feature_channels: int = 96,
        adapter_bottleneck: int = 32,
        decoder_channels: int = 64,
        use_adapter: bool = True,
    ) -> None:
        super().__init__()
        self.use_adapter = use_adapter
        self.backbone = SimpleBackbone(in_channels, base_channels, feature_channels)
        self.adapter = ErrorAwareResidualAdapter(
            channels=self.backbone.feature_channels,
            error_channels=1,
            error_embed_channels=8,
            bottleneck_channels=adapter_bottleneck,
        )
        self.decoder = UNetStyleDecoder(feature_channels, out_channels, decoder_channels)
 
    def freeze_base(self) -> None:
        """Freeze backbone and decoder for adapter-only feedback updates."""
        for module in (self.backbone, self.decoder):
            for param in module.parameters():
                param.requires_grad_(False)
        for param in self.adapter.parameters():
            param.requires_grad_(True)
 
    def unfreeze_all(self) -> None:
        for param in self.parameters():
            param.requires_grad_(True)
 
    def adapter_parameters(self):
        return [param for param in self.adapter.parameters() if param.requires_grad]
 
    def forward(self, x: Tensor, error_map: Optional[Tensor] = None, return_debug: bool = False):
        input_size = x.shape[-2:]
        features = self.backbone(x)
        refined_features, delta_features = self.adapter(features, error_map)
 
        logits = self.decoder(refined_features)
 
        if logits.shape[-2:] != input_size:
            logits = F.interpolate(logits, size=input_size, mode="bilinear", align_corners=False)
 
        if return_debug:
            return ForwardDebug(logits, features, refined_features, delta_features)
        return logits

## Losses and Metrics

Binary segmentation tensor contract:

```text
x:       [B, 1, H, W]
logits:  [B, 1, H, W]
G:       [B, 1, H, W]
```

The loss combines BCE and soft Dice. During feedback, we can optionally weight pixels with large correction errors more strongly.


In [6]:
def _check_binary_tensors(logits: Tensor, targets: Tensor) -> None:
    if logits.shape != targets.shape:
        raise ValueError(
            f"logits and targets must have same shape, got {tuple(logits.shape)} and {tuple(targets.shape)}"
        )
    if logits.ndim != 4 or logits.shape[1] != 1:
        raise ValueError(f"expected [B, 1, H, W] tensors, got {tuple(logits.shape)}")


def soft_dice_loss(logits: Tensor, targets: Tensor, eps: float = 1e-6) -> Tensor:
    _check_binary_tensors(logits, targets)
    probs = torch.sigmoid(logits)
    dims = (1, 2, 3)
    intersection = torch.sum(probs * targets, dim=dims)
    denominator = torch.sum(probs, dim=dims) + torch.sum(targets, dim=dims)
    dice = (2.0 * intersection + eps) / (denominator + eps)
    return 1.0 - dice.mean()


def bce_dice_loss(
    logits: Tensor,
    targets: Tensor,
    bce_weight: float = 1.0,
    dice_weight: float = 1.0,
    pixel_weight: Optional[Tensor] = None,
) -> Tensor:
    _check_binary_tensors(logits, targets)
    bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")

    if pixel_weight is not None:
        if pixel_weight.shape != logits.shape:
            pixel_weight = torch.broadcast_to(pixel_weight, logits.shape)
        bce = (bce * pixel_weight).sum() / pixel_weight.sum().clamp_min(1.0)
    else:
        bce = bce.mean()

    return bce_weight * bce + dice_weight * soft_dice_loss(logits, targets)


@torch.no_grad()
def binary_iou_score(logits: Tensor, targets: Tensor, threshold: float = 0.5, eps: float = 1e-6) -> Tensor:
    _check_binary_tensors(logits, targets)
    preds = (torch.sigmoid(logits) >= threshold).to(targets.dtype)
    dims = (1, 2, 3)
    intersection = torch.sum(preds * targets, dim=dims)
    union = torch.sum((preds + targets) > 0, dim=dims).to(targets.dtype)
    return ((intersection + eps) / (union + eps)).mean()


def evaluate_iou(
    model: DHFLSegmenter,
    loader: DataLoader,
    device: torch.device,
    max_batches: int = 20,
) -> float:

    model.eval()

    scores = []

    for batch_idx, (image, mask) in enumerate(loader):
        if batch_idx >= max_batches:
            break

        image = image.to(device)
        mask = mask.to(device)

        logits = model(image)

        scores.append(
            binary_iou_score(logits, mask)
        )

    return float(torch.stack(scores).mean().cpu())

## Synthetic Segmentation Dataset

This generates simple circles and rectangles so the architecture can be tested without downloading data.

In [8]:
class ShapeSegmentationDataset(Dataset):
    def __init__(
        self,
        samples: int = 256,
        image_size: int = 64,
        max_shapes: int = 3,
        seed: int = 0,
        noise_std: float = 0.15,
    ) -> None:
        self.samples = samples
        self.image_size = image_size
        self.max_shapes = max_shapes
        self.seed = seed
        self.noise_std = noise_std

        coords = torch.arange(image_size, dtype=torch.float32)
        self.yy, self.xx = torch.meshgrid(coords, coords, indexing="ij")

    def __len__(self) -> int:
        return self.samples

    def __getitem__(self, index: int):
        generator = torch.Generator().manual_seed(self.seed + index)
        size = self.image_size
        mask = torch.zeros((size, size), dtype=torch.bool)
        shape_count = int(torch.randint(1, self.max_shapes + 1, (1,), generator=generator).item())

        for _ in range(shape_count):
            kind = int(torch.randint(0, 2, (1,), generator=generator).item())
            if kind == 0:
                radius = float(torch.randint(size // 10, size // 4, (1,), generator=generator).item())
                cx = float(torch.randint(size // 5, 4 * size // 5, (1,), generator=generator).item())
                cy = float(torch.randint(size // 5, 4 * size // 5, (1,), generator=generator).item())
                shape = (self.xx - cx).pow(2) + (self.yy - cy).pow(2) <= radius**2
            else:
                width = int(torch.randint(size // 8, size // 3, (1,), generator=generator).item())
                height = int(torch.randint(size // 8, size // 3, (1,), generator=generator).item())
                x0 = int(torch.randint(0, size - width, (1,), generator=generator).item())
                y0 = int(torch.randint(0, size - height, (1,), generator=generator).item())
                shape = torch.zeros_like(mask)
                shape[y0 : y0 + height, x0 : x0 + width] = True

            mask = mask | shape

        mask_f = mask.float()
        gradient = 0.25 * (self.xx / max(size - 1, 1)) + 0.15 * (self.yy / max(size - 1, 1))
        noise = self.noise_std * torch.randn((size, size), generator=generator)
        image = (0.15 + gradient + 0.65 * mask_f + noise).clamp(0.0, 1.0)
        return image.unsqueeze(0), mask_f.unsqueeze(0)

In [10]:
@dataclass(frozen=True)
class FeedbackStepResult:
    loss: float
    iou_before: float
    iou_after: float
    mean_abs_error: float
    delta_abs_before: float
    delta_abs_after: float

def error_map_from_prediction(pred_logits: Tensor, corrected_mask: Tensor, detach: bool = True) -> Tensor:
    if pred_logits.shape != corrected_mask.shape:
        raise ValueError(
            f"pred_logits and corrected_mask must match, got {tuple(pred_logits.shape)} and {tuple(corrected_mask.shape)}"
        )

    probs = torch.sigmoid(pred_logits)
    if detach:
        probs = probs.detach()
        corrected_mask = corrected_mask.detach()

    return corrected_mask.float() - probs


def error_weight_map(error_map: Tensor, base: float = 1.0, gain: float = 4.0) -> Tensor:
    return base + gain * error_map.abs().detach()


def prepare_adapter_only_training(
    model: "DHFLSegmenter",
) -> dict:
    """Freeze backbone/decoder, return anchor weights snapshot.
 
    Returns
    -------
    anchor : dict[str, Tensor]
        A detached copy of the adapter's initial weights.
        Passing to every feedback_step call.
    """
    if not model.use_adapter:
        raise ValueError("adapter-only feedback requires use_adapter=True")
    model.freeze_base()

    anchor = {
        name: param.detach().clone()
        for name, param in model.adapter.named_parameters()
    }
    return anchor


def anchor_regularization_loss(
    model: "DHFLSegmenter",
    anchor: dict,
    anchor_strength: float,
) -> "Tensor":
    """Penalize drift from the adapter's initial weights.
 
    L_anchor = anchor_strength * Σ || w_current - w_anchor ||²
 
    When correction_signal → 0 (adapter has learned the correction),
    this term dominates and pulls weights back toward initialization,
    preventing further drift.
    """
    loss = torch.tensor(0.0, device=next(model.adapter.parameters()).device)
    for name, param in model.adapter.named_parameters():
        if name in anchor:
            loss = loss + (param - anchor[name]).pow(2).sum()
    return anchor_strength * loss


def feedback_step(
    model: "DHFLSegmenter",
    optimizer: "torch.optim.Optimizer",
    image: "Tensor",
    corrected_mask: "Tensor",
    anchor: dict,
    error_gain: float = 4.0,
    use_error_weights: bool = True,
    anchor_strength: float = 1e-3,
) -> "FeedbackStepResult":
    """One human-feedback update step with anchor regularization.
 
    Gradient behaviour
    ------------------
    Large E  → correction_signal ≈ 1, seg_loss dominates → adapter updates
    Small E  → correction_signal ≈ 0, seg_loss shrinks   → anchor pulls back
    E = 0    → correction_signal = 0, only anchor remains → weights return
                                                             to initialization
 
    The anchor_strength controls the trade-off:
      High → conservative, settles fast, good for simple corrections
      Low  → more freedom, slower settling, good for complex/ambiguous data
 
    Parameters
    ----------
    anchor : dict
        From prepare_adapter_only_training(). Must be passed every call.
    anchor_strength : float
        Weight of the anchor penalty. Default 1e-3.
        raise if adapter still drifts, lower if it under-learns.
    """
    model.train()
    corrected_mask = corrected_mask.float()
 
    # ── BEFORE UPDATE 
    with torch.no_grad():
        before = model(image, return_debug=True)
        logits_before = before.logits
 
        error_map = error_map_from_prediction(logits_before, corrected_mask)
 
        iou_before = binary_iou_score(logits_before, corrected_mask)
        delta_abs_before = before.delta_features.abs().mean()
 
        mean_abs_error = error_map.abs().mean()

        correction_signal = mean_abs_error / (mean_abs_error + 0.05)
 
    pixel_weight = error_weight_map(error_map, gain=error_gain) if use_error_weights else None
 
    # ── TRAIN STEP 
    optimizer.zero_grad(set_to_none=True)
 
    out = model(image, error_map=error_map, return_debug=True)
    logits = out.logits
    delta_features = out.delta_features
 
    seg_loss = bce_dice_loss(logits, corrected_mask, pixel_weight=pixel_weight)
    delta_reg = 1e-4 * delta_features.pow(2).mean()

    correction_loss = (seg_loss + delta_reg) * correction_signal
 
    anchor_loss = anchor_regularization_loss(model, anchor, anchor_strength)
 
    loss = correction_loss + anchor_loss
 
    loss.backward()
    optimizer.step()
 
    # ── AFTER UPDATE 
    with torch.no_grad():
        after = model(image, return_debug=True)
        logits_after = after.logits
        iou_after = binary_iou_score(logits_after, corrected_mask)
        delta_abs_after = after.delta_features.abs().mean()
 
    return FeedbackStepResult(
        loss=float(loss.detach().cpu()),
        iou_before=float(iou_before.cpu()),
        iou_after=float(iou_after.cpu()),
        mean_abs_error=float(mean_abs_error.cpu()),
        delta_abs_before=float(delta_abs_before.cpu()),
        delta_abs_after=float(delta_abs_after.cpu()),
    )

In [12]:
def maybe_update_anchor(
    anchor: dict,
    model: "DHFLSegmenter",
    current_iou: float,
    best_iou: float,
    threshold: float = 0.001,
) -> float:
    """Advance the anchor to current weights only if IoU is still improving.
 
    While the model keeps getting better, the anchor chases it upward →
    adapter is free to keep learning.
 
    Once improvement stalls, the anchor stays at the best weights seen →
    any further drift is penalised, producing a stable plateau.
 
    Returns updated best_iou.
    """
    if current_iou - best_iou > threshold:
        anchor.update({
            name: param.detach().clone()
            for name, param in model.adapter.named_parameters()
        })
        return current_iou
    return best_iou

## Training and Evaluation Helpers


In [14]:
def train_supervised_base(
    model: DHFLSegmenter,
    loader: DataLoader,
    steps: int,
    lr: float,
    device: torch.device,
) -> None:

    model.unfreeze_all()
    model.train()

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    batches = cycle(loader)

    for step in range(1, steps + 1):
        image, mask = next(batches)

        image = image.to(device)
        mask = mask.to(device)

        optimizer.zero_grad(set_to_none=True)

        # Base supervised training
        logits = model(image)

        loss = bce_dice_loss(logits, mask)

        loss.backward()
        optimizer.step()

        if step == 1 or step == steps or step % max(steps // 4, 1) == 0:
            print(
                f"base step {step:04d}/{steps} "
                f"loss={float(loss.detach().cpu()):.4f}"
            )

In [16]:
def run_smoke_tests() -> None:
    model = DHFLSegmenter(
        base_channels=8,
        feature_channels=24,
        adapter_bottleneck=8,
        decoder_channels=16,
        use_adapter=True,
    )
    x = torch.randn(2, 1, 64, 64)
 
    logits = model(x)
    debug = model(x, return_debug=True)
 
    assert logits.shape == (2, 1, 64, 64)
    assert debug.features.shape == (2, 24, 16, 16)
    assert debug.delta_features.shape == debug.features.shape
    assert debug.refined_features.shape == debug.features.shape
    assert torch.allclose(debug.refined_features, debug.features + debug.delta_features)
 
    print("Forward path check (inference, no error_map):")
    print("  F shape:       ", tuple(debug.features.shape))
    print("  delta shape:   ", tuple(debug.delta_features.shape))
    print("  F_prime shape: ", tuple(debug.refined_features.shape))
    print("  out shape:     ", tuple(debug.logits.shape))
    print("  mean |delta|:  ", float(debug.delta_features.abs().mean()))
    print("  note: delta=0 here because zero_init + no feedback yet (expected)")

    dataset = ShapeSegmentationDataset(samples=2, image_size=64, seed=123)
    image, mask = dataset[0]
    image = image.unsqueeze(0)
    mask = mask.unsqueeze(0)
 
    anchor = prepare_adapter_only_training(model)
    optimizer = torch.optim.Adam(model.adapter_parameters(), lr=1e-3)
    result = feedback_step(model, optimizer, image, mask, anchor=anchor)
 
    assert result.loss >= 0.0
    assert all(not param.requires_grad for param in model.backbone.parameters())
    assert all(not param.requires_grad for param in model.decoder.parameters())
    assert all(param.requires_grad for param in model.adapter.parameters())

    with torch.no_grad():
        debug_after = model(image, return_debug=True)
 
    print()
    print("smoke tests passed")
    print(f"  feedback iou before -> after: {result.iou_before:.4f} -> {result.iou_after:.4f}")
    print(f"  mean |delta| after 1 feedback step (inference, no error_map): "
          f"{float(debug_after.delta_features.abs().mean()):.6f}")
    print("  (delta should be > 0 now — adapter active at inference)")
 
 
run_smoke_tests()

Forward path check (inference, no error_map):
  F shape:        (2, 24, 16, 16)
  delta shape:    (2, 24, 16, 16)
  F_prime shape:  (2, 24, 16, 16)
  out shape:      (2, 1, 64, 64)
  mean |delta|:   0.0
  note: delta=0 here because zero_init + no feedback yet (expected)

smoke tests passed
  feedback iou before -> after: 0.0217 -> 0.0217
  mean |delta| after 1 feedback step (inference, no error_map): 0.001113
  (delta should be > 0 now — adapter active at inference)


## Correction Count Sweep

This section increases the number of feedback/correction updates and reports
whether the adapter improves.

Important interpretation:

- `feedback IoU` measures improvement on the correction distribution.
- `test IoU` measures generalization to held-out synthetic samples.
- If feedback IoU improves but test IoU does not, the adapter may be overfitting
  to corrections or the feedback set may not cover the test distribution.


In [20]:
if RUN_CORRECTION_SWEEP:
    torch.manual_seed(7)
 
    BASE_STEPS            = 30
    CORRECTION_STEPS_LIST = [0, 10, 30, 80, 160, 320, 640, 1280]
    BATCH_SIZE            = 8
    ANCHOR_STRENGTH       = 5e-3  
 
    train_ds         = ShapeSegmentationDataset(samples=128, image_size=64, seed=7)
    feedback_ds      = ShapeSegmentationDataset(samples=256, image_size=64, seed=10_007)
    test_ds          = ShapeSegmentationDataset(samples=128, image_size=64, seed=20_007)
 
    train_loader         = DataLoader(train_ds,    batch_size=BATCH_SIZE, shuffle=True)
    feedback_loader      = DataLoader(feedback_ds, batch_size=BATCH_SIZE, shuffle=True)
    feedback_eval_loader = DataLoader(feedback_ds, batch_size=BATCH_SIZE, shuffle=False)
    test_loader          = DataLoader(test_ds,     batch_size=BATCH_SIZE, shuffle=False)
 
    base_model = DHFLSegmenter(
        base_channels=8, feature_channels=24,
        adapter_bottleneck=8, decoder_channels=16, use_adapter=True,
    ).to(DEVICE)
 
    print("Training base model...")
    train_supervised_base(base_model, train_loader, steps=BASE_STEPS, lr=1e-3, device=DEVICE)
 
    base_state        = copy.deepcopy(base_model.state_dict())
    base_feedback_iou = evaluate_iou(base_model, feedback_eval_loader, DEVICE)
    base_test_iou     = evaluate_iou(base_model, test_loader, DEVICE)
 
    print(f"\nBefore feedback:\n  feedback IoU: {base_feedback_iou:.4f}\n  test IoU: {base_test_iou:.4f}\n")
 
    results = []
 
    for correction_steps in CORRECTION_STEPS_LIST:
        model = DHFLSegmenter(
            base_channels=8, feature_channels=24,
            adapter_bottleneck=8, decoder_channels=16, use_adapter=True,
        ).to(DEVICE)
        model.load_state_dict(base_state)
 
        anchor    = prepare_adapter_only_training(model)
        optimizer = torch.optim.Adam(model.adapter_parameters(), lr=1e-3, weight_decay=1e-5)
        batches   = cycle(feedback_loader)
        best_iou  = base_feedback_iou 
 
        last_result = None
        for step in range(1, correction_steps + 1):
            image, corrected_mask = next(batches)
            image, corrected_mask = image.to(DEVICE), corrected_mask.to(DEVICE)
 
            last_result = feedback_step(
                model, optimizer, image, corrected_mask,
                anchor=anchor,
                error_gain=4.0,
                use_error_weights=True,
                anchor_strength=ANCHOR_STRENGTH, 
            )
 
            if step % 20 == 0:
                current_iou = evaluate_iou(model, feedback_eval_loader, DEVICE)
                best_iou    = maybe_update_anchor(anchor, model, current_iou, best_iou)
 
            if step in {1, correction_steps} or step % 40 == 0:
                print(
                    f"corrections={correction_steps:03d} "
                    f"step={step:03d}/{correction_steps:03d} "
                    f"loss={last_result.loss:.4f} "
                    f"batch_iou={last_result.iou_before:.4f}->{last_result.iou_after:.4f} "
                    f"|delta|={last_result.delta_abs_before:.6f}->{last_result.delta_abs_after:.6f}"
                )
 
        feedback_iou = evaluate_iou(model, feedback_eval_loader, DEVICE)
        test_iou     = evaluate_iou(model, test_loader, DEVICE)
 
        results.append({
            "correction_steps":        correction_steps,
            "approx_corrected_images": correction_steps * BATCH_SIZE,
            "feedback_iou":            feedback_iou,
            "test_iou":                test_iou,
            "feedback_delta":          feedback_iou - base_feedback_iou,
            "test_delta":              test_iou - base_test_iou,
        })
 
    print("\nCorrection sweep summary (adaptive anchor, strength=5e-3)")
    print("steps | approx corrected images | feedback IoU | test IoU | feedback delta | test delta")
    for row in results:
        print(
            f"{row['correction_steps']:5d} | "
            f"{row['approx_corrected_images']:23d} | "
            f"{row['feedback_iou']:.4f}       | "
            f"{row['test_iou']:.4f}   | "
            f"{row['feedback_delta']:+.4f}         | "
            f"{row['test_delta']:+.4f}"
        )

Training base model...
base step 0001/30 loss=1.6652
base step 0007/30 loss=1.3703
base step 0014/30 loss=1.2791
base step 0021/30 loss=1.1514
base step 0028/30 loss=1.1641
base step 0030/30 loss=1.2141

Before feedback:
  feedback IoU: 0.5215
  test IoU: 0.5473

corrections=010 step=001/010 loss=1.1992 batch_iou=0.4383->0.4387 |delta|=0.008623->0.008682
corrections=010 step=010/010 loss=1.0791 batch_iou=0.5775->0.5778 |delta|=0.009570->0.009743
corrections=030 step=001/030 loss=1.0550 batch_iou=0.5799->0.5802 |delta|=0.008616->0.008715
corrections=030 step=030/030 loss=1.0635 batch_iou=0.5737->0.5737 |delta|=0.021801->0.022557
corrections=080 step=001/080 loss=1.0512 batch_iou=0.5920->0.5922 |delta|=0.008605->0.008851
corrections=080 step=040/080 loss=1.2332 batch_iou=0.4100->0.4098 |delta|=0.029102->0.029682
corrections=080 step=080/080 loss=1.1569 batch_iou=0.4531->0.4531 |delta|=0.060296->0.060617
corrections=160 step=001/160 loss=1.1549 batch_iou=0.4796->0.4796 |delta|=0.008616->0

## Baseline Switches for Adapter / No Adapter Testing

Use these toggles for the next comparison after the correction sweep works.

The key scientific question is:

> Does adapter-only feedback improve more than ordinary training noise or a
> no-adapter baseline?


In [22]:
EXPERIMENT_GRID = [
    {
        "name": "no_adapter_base",
        "use_adapter": False,
        "do_feedback": False,
        "use_error_weights": False,
    },
    {
        "name": "adapter_base",
        "use_adapter": True,
        "do_feedback": False,
        "use_error_weights": False,
    },
    {
        "name": "adapter_feedback_plain",
        "use_adapter": True,
        "do_feedback": True,
        "use_error_weights": False,
    },
    {
        "name": "adapter_feedback_error_weighted",
        "use_adapter": True,
        "do_feedback": True,
        "use_error_weights": True,
    },
]

EXPERIMENT_GRID

[{'name': 'no_adapter_base',
  'use_adapter': False,
  'do_feedback': False,
  'use_error_weights': False},
 {'name': 'adapter_base',
  'use_adapter': True,
  'do_feedback': False,
  'use_error_weights': False},
 {'name': 'adapter_feedback_plain',
  'use_adapter': True,
  'do_feedback': True,
  'use_error_weights': False},
 {'name': 'adapter_feedback_error_weighted',
  'use_adapter': True,
  'do_feedback': True,
  'use_error_weights': True}]

In [24]:
def run_experiment_grid(
    experiment_grid: list,
    base_steps: int = 30,
    correction_steps: int = 80,  
    batch_size: int = 8,
    anchor_strength: float = 5e-3,
    device: torch.device = DEVICE,
) -> None:
    torch.manual_seed(42)
 
    train_ds        = ShapeSegmentationDataset(samples=128, image_size=64, seed=42)
    feedback_ds     = ShapeSegmentationDataset(samples=256, image_size=64, seed=10_042)
    test_ds         = ShapeSegmentationDataset(samples=128, image_size=64, seed=20_042)
 
    train_loader        = DataLoader(train_ds,    batch_size=batch_size, shuffle=True)
    feedback_loader     = DataLoader(feedback_ds, batch_size=batch_size, shuffle=True)
    feedback_eval_loader = DataLoader(feedback_ds, batch_size=batch_size, shuffle=False)
    test_loader         = DataLoader(test_ds,     batch_size=batch_size, shuffle=False)

    print("Training shared base model...")
    shared_base = DHFLSegmenter(
        base_channels=8, feature_channels=24,
        adapter_bottleneck=8, decoder_channels=16, use_adapter=True,
    ).to(device)
    train_supervised_base(shared_base, train_loader, steps=base_steps, lr=1e-3, device=device)
    base_state = copy.deepcopy(shared_base.state_dict())
 
    base_feedback_iou = evaluate_iou(shared_base, feedback_eval_loader, device)
    base_test_iou     = evaluate_iou(shared_base, test_loader, device)
    print(f"\nBase model — feedback IoU: {base_feedback_iou:.4f}  test IoU: {base_test_iou:.4f}\n")
 
    grid_results = []
 
    for exp in experiment_grid:
        name             = exp["name"]
        use_adapter      = exp["use_adapter"]
        do_feedback      = exp["do_feedback"]
        use_error_weights = exp["use_error_weights"]
 
        model = DHFLSegmenter(
            base_channels=8, feature_channels=24,
            adapter_bottleneck=8, decoder_channels=16,
            use_adapter=use_adapter,
        ).to(device)

        model.load_state_dict(base_state, strict=False)
 
        if do_feedback and use_adapter:
            anchor    = prepare_adapter_only_training(model)
            optimizer = torch.optim.Adam(model.adapter_parameters(), lr=1e-3, weight_decay=1e-5)
            batches   = cycle(feedback_loader)
            best_iou  = base_feedback_iou
 
            for step in range(1, correction_steps + 1):
                image, corrected_mask = next(batches)
                image, corrected_mask = image.to(device), corrected_mask.to(device)
 
                feedback_step(
                    model, optimizer, image, corrected_mask,
                    anchor=anchor,
                    error_gain=4.0,
                    use_error_weights=use_error_weights,
                    anchor_strength=anchor_strength,
                )

                if step % 20 == 0:
                    current_iou = evaluate_iou(model, feedback_eval_loader, device)
                    best_iou    = maybe_update_anchor(anchor, model, current_iou, best_iou)
 
        feedback_iou = evaluate_iou(model, feedback_eval_loader, device)
        test_iou     = evaluate_iou(model, test_loader, device)
 
        grid_results.append({
            "name":           name,
            "feedback_iou":   feedback_iou,
            "test_iou":       test_iou,
            "feedback_delta": feedback_iou - base_feedback_iou,
            "test_delta":     test_iou - base_test_iou,
        })
 
        print(f"[{name}]  feedback={feedback_iou:.4f} ({feedback_iou-base_feedback_iou:+.4f})  "
              f"test={test_iou:.4f} ({test_iou-base_test_iou:+.4f})")
 
    print("\nExperiment grid summary")
    print(f"{'experiment':<35} | feedback IoU | test IoU | feedback Δ | test Δ")
    print("-" * 80)
    for r in grid_results:
        print(f"{r['name']:<35} | {r['feedback_iou']:.4f}       | {r['test_iou']:.4f}   | "
              f"{r['feedback_delta']:+.4f}      | {r['test_delta']:+.4f}")
 
 
run_experiment_grid(EXPERIMENT_GRID, correction_steps=80, anchor_strength=5e-3)

Training shared base model...
base step 0001/30 loss=1.5472
base step 0007/30 loss=1.3333
base step 0014/30 loss=1.2579
base step 0021/30 loss=1.2135
base step 0028/30 loss=1.1709
base step 0030/30 loss=1.1833

Base model — feedback IoU: 0.4933  test IoU: 0.5121

[no_adapter_base]  feedback=0.4933 (+0.0000)  test=0.5121 (+0.0000)
[adapter_base]  feedback=0.4933 (+0.0000)  test=0.5121 (+0.0000)
[adapter_feedback_plain]  feedback=0.4997 (+0.0064)  test=0.5169 (+0.0048)
[adapter_feedback_error_weighted]  feedback=0.5047 (+0.0114)  test=0.5218 (+0.0097)

Experiment grid summary
experiment                          | feedback IoU | test IoU | feedback Δ | test Δ
--------------------------------------------------------------------------------
no_adapter_base                     | 0.4933       | 0.5121   | +0.0000      | +0.0000
adapter_base                        | 0.4933       | 0.5121   | +0.0000      | +0.0000
adapter_feedback_plain              | 0.4997       | 0.5169   | +0.0064      | +